In [0]:
# Cell 1
%pip install langgraph langchain langchain-community --quiet
%pip install --upgrade databricks-langchain langchain-community langchain databricks-sql-connector


In [0]:
%pip install langchain-core databricks-langchain langgraph-supervisor mlflow plotly

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

In [0]:
# Cell 2
import json
import time
from typing import TypedDict, List, Dict, Any, Callable, Optional
from functools import wraps
from uuid import uuid4
import pandas as pd
# Databricks LLM & client
from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
# Optional: from databricks_langchain.genie import GenieAgent

# LangGraph state graph
from langgraph.graph import StateGraph, END

# LangChain message helper for LLM calls
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Spark & plotting
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

import matplotlib.pyplot as plt

# Initialize Databricks Function client (uses workspace credentials)
client = DatabricksFunctionClient()
set_uc_function_client(client)

# CONFIG - change these as needed
USE_LLM_SUPERVISOR = True            # set True to let LLM decide next agent
USE_LLM_SQL_AGENT = True             # set True to let LLM generate SQL
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"  # e.g. "databricks-claude-3-7-sonnet"
EXECUTE_SQL = True                   # whether to actually run generated SQL (requires table present)
TARGET_TABLE = "demo.retail_media"        # default target table (update to your table)
SQL_RETRY_COUNT = 2
SQL_RETRY_DELAY_SECONDS = 2

# Initialize LLM client only if needed
llm: Optional[ChatDatabricks] = None
if (USE_LLM_SUPERVISOR or USE_LLM_SQL_AGENT):
    if not LLM_ENDPOINT_NAME or "<" in LLM_ENDPOINT_NAME:
        raise ValueError("Set LLM_ENDPOINT_NAME to your Databricks endpoint to enable LLM features.")
    llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
    print(f"✅ ChatDatabricks initialized: {LLM_ENDPOINT_NAME}")

print("✅ Initialization complete.")


In [0]:
import json
import re
from uuid import uuid4
from typing import Dict, Any, Generator
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langgraph.graph.state import CompiledStateGraph
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)

# -----------------------------
# CONFIG
# -----------------------------
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
TABLE_NAME = "demo.retail_media"
MAX_SQL_ROWS = 2000

# Initialize databricks function client and LLM
client = DatabricksFunctionClient()
set_uc_function_client(client)
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

# Simple prompt builder
def make_prompt(template: str, question: str) -> str:
    return template.replace("{question}", question)

# SQL prompt template
SQL_PROMPT_TEXT = """
You are an expert SQL generator for Databricks Delta tables.

Available table: `{table_name}` with columns:
DATE, ZONE, REGION, COUNTRY, RETAIL_CHANNEL, RETAILER_NAME, MANUFACTURER, PRODUCT_FAMILY, SPECIES, BRAND, SUB_BRAND, SKU_NAME, MARKETING_CHANNEL, CAMPAIGN_NAME, METRIC, VALUE

Rules:
- Return a single VALID Databricks SQL SELECT statement, no explanation, no markdown fences.
- Use uppercase for SQL keywords (SELECT, FROM, WHERE, GROUP BY, ORDER BY).
- Use ISO date format 'YYYY-MM-DD' when filtering by date.
- If ambiguous, produce a conservative aggregation (SUM(VALUE) with GROUP BY).
- Always include an ORDER BY.
- Do NOT include destructive statements.

User question:
{question}
""".strip().replace("{table_name}", TABLE_NAME)
CHART_PLAN_PROMPT = """
You are a visualization planner. Given a short user request and a small JSON sample (first up to 5 rows),
return ONLY a single JSON object (no extra text) following this schema:

{
  "chart_type": "LINE" | "BAR" | "PIE" | "HEATMAP",
  "x": "<column name or null>",
  "y": ["<col1>", "<col2>", ...],
  "aggregation": "sum" | "avg" | "count" | null,
  "date_construction": {"type":"year_month" | "date_col" | null, "year_col":"YEAR"?, "month_col":"MONTH"?, "date_col":"DATE"?},
  "title": "<short chart title>"
}

Rules:
- Choose chart_type based on the question and sample.
- Use column names exactly as they appear in the sample (case-sensitive).
- If a date axis is needed and only YEAR+MONTH exist, set date_construction.type = "year_month" and provide year_col/month_col.
- Keep y as a list even if one element.
- If uncertain, prefer LINE for time series and GROUP BY SUM.
"""
CHART_CODE_PROMPT_TEMPLATE = """
You are a Python/Plotly code generator. You will receive:
- A visualization plan as a JSON object (chart_type, x, y, aggregation, date_construction, title).
- A pandas DataFrame named `df` already loaded with the full query results.

Produce ONLY runnable Python code (no prose, no fences) that:
- imports pandas and plotly.express (and plotly.graph_objects if needed),
- constructs a datetime column if date_construction.type == "year_month",
- performs aggregation if aggregation is non-null (use df.groupby([...])[y].agg(aggregation).reset_index()),
- creates a Plotly Figure assigned to variable `fig`,
- calls `fig.show()` as the last statement.

Use defensive checks: `if 'COL' in df.columns` before using a column.
Keep code compact.
"""



# Data analysis prompt template
DATA_ANALYSIS_PROMPT_TEXT = """
You are a data analysis expert. You will receive:
- Table schema (columns)
- A small JSON sample of the query results (first up to 5 rows)
- The original user question

Your task — produce ONE single plain-text response (no code, no JSON, no markdown fences) that is concise (<= ~300 words) and contains the following sections, in this order:

1) ONE-LINE SUMMARY: one sentence that answers or summarizes the main finding relative to the user's question.
2) KEY OBSERVATIONS: 3 short bullets (one sentence each) about distributions, trends, outliers, missing values, or anything directly visible in the sample.
3) SUGGESTED NEXT STEPS: 3 short, concrete, prioritized actions (SQL adjustments, additional slices to check, or visualization choices). Start each step with "1.", "2.", "3.".
4) POTENTIAL RISKS / DATA QUALITY NOTES: up to 2 brief bullets about issues to watch for (e.g., nulls, aggregated metrics, mixed units).
5) RECOMMENDED VISUAL: a single short line recommending the best chart (LINE / BAR / HEATMAP / PIE) and what should be on X and Y.

Constraints:
- Plain text only. Do NOT include headings or markdown formatting.
- Keep it compact and actionable (aim for ~150-300 words).
- If the sample is too small to be certain, state uncertainty in the ONE-LINE SUMMARY and recommend a clarification or additional slice.
"""


# LLM call helper
def call_llm_return_text(prompt: str) -> str:
    try:
        if hasattr(llm, "generate"):
            out = llm.generate([{"role":"user","content": prompt}])
            if hasattr(out, "generations"):
                gens = out.generations
                if isinstance(gens, (list, tuple)) and len(gens) > 0:
                    g0 = gens[0]
                    if isinstance(g0, (list, tuple)):
                        return getattr(g0[0], "text", str(g0[0]))
                    else:
                        return getattr(g0, "text", str(g0))
            return str(out)
    except Exception:
        pass
    try:
        if callable(llm):
            out = llm(prompt)
            if isinstance(out, str):
                return out
            if hasattr(out, "content"):
                return out.content
            if hasattr(out, "text"):
                return out.text
            return str(out)
    except Exception:
        pass
    for fn in ("invoke", "run", "predict", "chat", "generate_text", "complete"):
        if hasattr(llm, fn):
            try:
                out = getattr(llm, fn)(prompt)
                if isinstance(out, str):
                    return out
                if hasattr(out, "content"):
                    return out.content
                if hasattr(out, "text"):
                    return out.text
                return str(out)
            except Exception:
                continue
    raise RuntimeError("Unable to call ChatDatabricks LLM; inspect dir(llm).")

# Safe SQL runner
def run_sql(query: str, max_rows: int = MAX_SQL_ROWS) -> Dict[str, Any]:
    if not isinstance(query, str):
        raise ValueError("query must be a string")
    q_upper = query.upper()
    forbidden = ["DROP ", "DELETE ", "TRUNCATE ", "ALTER ", "SHUTDOWN", "GRANT ", "REVOKE ", "CREATE TABLE", "CREATE DATABASE"]
    for kw in forbidden:
        if kw in q_upper:
            raise ValueError(f"Refusing to run query containing forbidden keyword: {kw.strip()}")
    if "SELECT" not in q_upper:
        raise ValueError("Only SELECT queries are allowed.")
    if "LIMIT" not in q_upper:
        query = f"SELECT * FROM ({query.rstrip(';')})"
    df = spark.sql(query)
    pdf = df.toPandas()
    return {"query": query, "rows": json.loads(pdf.to_json(orient="records", date_format="iso")), "rowcount": len(pdf), "columns": list(pdf.columns)}

# SQL generation agent
def sql_agent_call(user_question: str) -> str:
    prompt = make_prompt(SQL_PROMPT_TEXT, user_question)
    out = call_llm_return_text(prompt)
    up = out.upper()
    if "SELECT" in up:
        idx = up.find("SELECT")
        cand = out[idx:]
        cand = re.sub(r"```$", "", cand).strip()
        return cand
    return out.strip()

# Chart/visualization agent
def chart_agent_call(user_question: str, sample_json: str) -> str:
    """
    1) Ask LLM for a JSON visualization plan.
    2) Validate JSON plan and ask LLM to generate Python code from that plan.
    3) Return the Python code. If anything fails, return deterministic fallback code.
    """
    # 1) plan
    plan_prompt = CHART_PLAN_PROMPT + "\n\nUser question:\n" + user_question + "\n\nSample data:\n" + sample_json
    raw_plan = call_llm_return_text(plan_prompt).strip()

    # try to extract JSON object from the model output
    plan_text = raw_plan
    # common model behavior: sometimes wrapped in markdown or triple backticks -> strip fences
    plan_text = re.sub(r"^\\s*```(?:json)?\\s*", "", plan_text, flags=re.IGNORECASE)
    plan_text = re.sub(r"\\s*```\\s*$", "", plan_text, flags=re.IGNORECASE)
    plan_text = plan_text.strip()

    try:
        plan = json.loads(plan_text)
        print(plan)
    except Exception:
        # fallback: ask model again for plain JSON only (shorter prompt)
        try:
            retry_prompt = "Return ONLY a single JSON object with keys: chart_type,x,y,aggregation,date_construction,title. Use the sample columns exactly. Question: " + user_question
            plan = json.loads(call_llm_return_text(retry_prompt))
        except Exception:
            # final fallback: create a conservative plan
            try:
                sample_df = pd.DataFrame(json.loads(sample_json))
            except Exception:
                sample_df = pd.DataFrame()
            # choose first numeric-ish columns as y
            y_cols = [c for c in sample_df.columns if c.upper() not in ("DATE","YEAR","MONTH")][:2]
            if not y_cols and len(sample_df.columns) > 0:
                y_cols = [sample_df.columns[0]]
            plan = {
                "chart_type": "LINE",
                "x": "DATE" if "DATE" in sample_df.columns else None,
                "y": y_cols or [],
                "aggregation": "sum",
                "date_construction": {"type": "date_col" if "DATE" in sample_df.columns else None},
                "title": user_question[:80]
            }

    # 2) generate code from plan
    code_prompt = CHART_CODE_PROMPT_TEMPLATE + "\n\nVisualization plan JSON:\n" + json.dumps(plan) + "\n\nUser question:\n" + user_question
    raw_code = call_llm_return_text(code_prompt)

    # strip fences
    code = re.sub(r"^\\s*```(?:python)?\\s*", "", raw_code, flags=re.IGNORECASE)
    code = re.sub(r"\\s*```\\s*$", "", code, flags=re.IGNORECASE)
    code = code.strip()

    # quick sanity checks
    if "fig" not in code or "fig.show" not in code:
        # try one more time with an explicit instruction that code must end with fig.show()
        code = call_llm_return_text(code_prompt + "\nEnsure the code assigns to `fig` and ends with `fig.show()`.")
        code = re.sub(r"^\\s*```(?:python)?\\s*", "", code, flags=re.IGNORECASE)
        code = re.sub(r"\\s*```\\s*$", "", code, flags=re.IGNORECASE)
        code = code.strip()

    # validate by compiling
    try:
        compile(code, "<chart_code>", "exec")
        return code
    except SyntaxError:
        # final fallback deterministic code using plan
        try:
            sample_df = pd.DataFrame(json.loads(sample_json))
        except Exception:
            sample_df = pd.DataFrame()
        return fallback_chart_code(user_question, sample_df, chart_choice=plan.get("chart_type", "LINE"))
    
def fallback_chart_code(question: str, df_sample: pd.DataFrame, chart_choice: str = "LINE") -> str:
    cols = list(df_sample.columns) if isinstance(df_sample, pd.DataFrame) else []
    code_lines = ["import pandas as pd", "import plotly.express as px", "df = df.copy()"]
    if "DATE" in cols:
        code_lines.append("df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')")
    elif "YEAR" in (c.upper() for c in cols) and "MONTH" in (c.upper() for c in cols):
        code_lines.append("df['DATE'] = pd.to_datetime(df[['YEAR','MONTH']].assign(DAY=1))")
    # choose y cols
    y_candidates = [c for c in cols if c.upper() not in ("DATE","YEAR","MONTH")]
    y_cols = y_candidates[:2] if y_candidates else (cols[:1] if cols else ["VALUE"])
    code_lines.append(f"fig = px.line(df, x='DATE', y={y_cols!r})")
    code_lines.append("fig.update_layout(autosize=True)")
    code_lines.append("fig.show()")
    return "\n".join(code_lines)

# Data analysis agent
def data_analysis_agent_call(question: str, schema: str, sample_json: str) -> str:
    # Build prompt
    prompt = DATA_ANALYSIS_PROMPT_TEXT.format(schema=schema, sample_json=sample_json, question=question)
    # Append a final guard to ensure single plain-text reply
    prompt += "\n\nRemember: return ONE single plain-text response only (no code, no lists beyond what is asked)."

    raw = call_llm_return_text(prompt)

    # Post-process: normalize whitespace, collapse duplicate newlines, trim length
    text = re.sub(r"\r\n", "\n", raw).strip()
    text = re.sub(r"\n{3,}", "\n\n", text)  # collapse >2 newlines
    # If the model returned numbered lists or headings, keep them but ensure plain text
    # Enforce approximate length (truncate politely to last full sentence under 350 words)
    words = text.split()
    if len(words) > 400:
        # truncate to ~350 words preserving sentence boundary
        truncated = " ".join(words[:350])
        # try to end at last period
        last_dot = truncated.rfind(".")
        if last_dot != -1 and last_dot > int(len(truncated) * 0.5):
            text = truncated[: last_dot + 1]
        else:
            text = truncated + " ..."
    return text.strip()


# Create agent objects
agent_objects = [
    create_agent(llm, tools=[], name="sql-generator-agent"),
    create_agent(llm, tools=[], name="chart-generator-agent"),
    create_agent(llm, tools=[], name="data-analysis-agent"),
]

# Supervisor prompt with three subagents
supervisor_prompt = f"""
You are a supervisor responsible for coordinating subagents to answer the user's question.
Available subagents:
- sql-generator-agent: generates a Databricks SQL SELECT statement for table `{TABLE_NAME}`.
- chart-generator-agent: given a small JSON sample and the user's question, returns executable Plotly Python code (variable 'fig' and fig.show()).
- data-analysis-agent: analyzes data schema and sample to produce relevant insights and instructions.

Supervisor instructions:
1) Read the user's request.
2) Decide which subagent(s) to call and in what order.
3) Ask the sql-generator-agent to produce the SQL when needed.
4) After SQL is executed, decide to call data-analysis-agent for insights or chart-generator-agent for visualization.
5) Prefer safe, read-only operations.
6) Return a short orchestration trace and final result pointer.
"""

compiled_supervisor: CompiledStateGraph = create_supervisor(
    agents=agent_objects,
    model=llm,
    prompt=supervisor_prompt,
    add_handoff_messages=False,
    output_mode="full_history"
).compile()

class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, compiled: CompiledStateGraph):
        self.compiled = compiled
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [event.item for event in self.predict_stream(request) if event.type == "response.output_item.done"]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)
    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        first = True
        seen = set()
        for _, events in self.compiled.stream({"messages": cc_msgs}, stream_mode=["updates"]):
            new_msgs = [msg for v in events.values() for msg in v.get("messages", []) if msg.id not in seen]
            if first:
                seen.update(msg.id for msg in new_msgs[: len(cc_msgs)])
                new_msgs = new_msgs[len(cc_msgs):]
                first = False
            else:
                seen.update(msg.id for msg in new_msgs)
                node_name = tuple(events.keys())[0]
                yield ResponsesAgentStreamEvent(type="response.output_item.done", item=self.create_text_output_item(text=f"<name>{node_name}</name>", id=str(uuid4())))
            if len(new_msgs) > 0:
                yield from output_to_responses_items_stream(new_msgs)

SUPERVISOR_AGENT = LangGraphResponsesAgent(compiled_supervisor)

# Updated agentic pipeline integrating all three agents
# def nl_to_chart_and_analysis_agentic(user_question: str):
#     trace = {"supervisor_prompt": supervisor_prompt, "actions": []}

#     # Ask supervisor to pick the first subagent
#     supervisor_query = f"User question: {user_question}\n\nPlease respond with which subagent to call first: one of [sql-generator-agent, chart-generator-agent, data-analysis-agent]. Only return the agent name."
#     try:
#         sup_resp = call_llm_return_text(supervisor_query)
#         chosen = sup_resp.strip().splitlines()[0].strip()
#         trace["actions"].append({"supervisor_decision": chosen, "raw": sup_resp})
#     except Exception as e:
#         chosen = "sql-generator-agent"
#         trace["actions"].append({"supervisor_decision": chosen, "error": str(e)})

#     # Call SQL generation agent - mandatory to get data
#     sql_text = sql_agent_call(user_question)
#     trace["actions"].append({"sql_generated_raw": sql_text})
#     up = sql_text.upper()
#     idx = up.find("SELECT")
#     if idx != -1:
#         sql_text = sql_text[idx:]
#     generated_sql = sql_text.strip()
#     trace["actions"].append({"generated_sql": generated_sql})
#     print("=== Generated SQL ===")
#     print(generated_sql)

#     # Run SQL query
#     sql_result = run_sql(generated_sql, max_rows=MAX_SQL_ROWS)
#     rows = sql_result.get("rows", [])
#     df = pd.DataFrame(rows)
#     trace["actions"].append({"executed_rowcount": sql_result.get("rowcount", 0)})

#     if df.empty:
#         print("Query returned 0 rows.")
#         return {"trace": trace, "sql": generated_sql, "df": df, "fig": None, "analysis": None}

#     schema_str = json.dumps({"columns": sql_result.get("columns")}, indent=2)
#     sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str, indent=2)

#     if chosen == "data-analysis-agent":
#         analysis_results = data_analysis_agent_call(user_question, schema_str, sample_json)
#         trace["actions"].append({"data_analysis_output": analysis_results})
#         # Print a clear human-readable output (single refined response)
#         print("\n=== Data Analysis (Refined) ===\n")
#         print(analysis_results)
#         print("\n=== End of Analysis ===\n")
#         return {"trace": trace, "sql": generated_sql, "df": df, "fig": None, "analysis": analysis_results}


#     else:
#         chart_decision_prompt = f"User question: {user_question}\nSample data (first 5 rows):\n{sample_json}\n\nWhich chart type should be used? Choose one of: LINE, BAR, PIE, HEATMAP. Return only the word."
#         try:
#             chart_choice_raw = call_llm_return_text(chart_decision_prompt)
#             chart_choice = chart_choice_raw.strip().splitlines()[0].strip().upper()
#         except Exception:
#             chart_choice = "LINE" if any(c.lower() == "date" for c in df.columns) else "BAR"
#         trace["actions"].append({"chart_choice": chart_choice})

#         chart_code = chart_agent_call(user_question, sample_json)
#         trace["actions"].append({"chart_code_snippet": chart_code[:300] if chart_code else None})

#         fig = None
#         if chart_code:
#     # Prepare environment: preload commonly-imported modules and a DataFrame copy
#             local_df = df.copy()
#             safe_globals = {
#                 "pd": pd,
#                 "px": px,
#                 "go": go,
#                 "df": local_df,
#                 # minimal safe builtins for simple operations (no __import__)
#                 "__builtins__": {"len": len, "range": range, "min": min, "max": max, "sum": sum},
#             }

#             # Strip import lines from the model code because we already provide pd/px/go
#             # This avoids needing __import__ inside the sandbox.
#             cleaned_code_lines = []
#             for line in chart_code.splitlines():
#                 # skip import statements (import ... or from ... import ...)
#                 if re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", line):
#                     continue
#                 cleaned_code_lines.append(line)
#             cleaned_code = "\n".join(cleaned_code_lines).strip()

#             # If cleaning removed everything (unlikely), generate fallback
#             if not cleaned_code:
#                 trace["actions"].append({"chart_exec_error": "All code lines were imports; running fallback."})
#                 try:
#                     fallback_code = fallback_chart_code(user_question, df.head(5), chart_choice if 'chart_choice' in locals() else "LINE")
#                     cleaned_code = "\n".join([ln for ln in fallback_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)])
#                 except Exception as e:
#                     trace["actions"].append({"fallback_chart_error": str(e)})
#                     cleaned_code = ""

#             fig = None
#             try:
#                 # compile first to catch syntax errors
#                 compiled = compile(cleaned_code, "<chart_code>", "exec")
#                 exec(compiled, safe_globals, {})
#                 fig = safe_globals.get("fig") or None
#                 if fig is not None:
#                     try:
#                         fig.show()
#                     except Exception:
#                         try:
#                             display(fig)
#                         except Exception:
#                             pass
#             except Exception as e:
#                 trace["actions"].append({"chart_exec_error": str(e)})
#                 # fallback attempt: deterministic code (we will strip imports similarly)
#                 try:
#                     fallback_code = fallback_chart_code(user_question, df.head(5), chart_choice if 'chart_choice' in locals() else "LINE")
#                     fallback_lines = [ln for ln in fallback_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
#                     fallback_clean = "\n".join(fallback_lines)
#                     compiled_fb = compile(fallback_clean, "<fallback_chart>", "exec")
#                     exec(compiled_fb, safe_globals, {})
#                     fig = safe_globals.get("fig") or None
#                     if fig is not None:
#                         try:
#                             fig.show()
#                         except Exception:
#                             try:
#                                 display(fig)
#                             except Exception:
#                                 pass
#                 except Exception as e2:
#                     trace["actions"].append({"fallback_chart_error": str(e2)})
#                     fig = None

#         # Optional: fallback heuristics for charting can be added here as before

#         return {"trace": trace, "sql": generated_sql, "df": df, "fig": fig, "analysis": None}
def nl_to_chart_and_analysis_agentic_supervised(user_question: str) -> Dict[str, Any]:
    """
    Use compiled_supervisor to decide which subagents to run and in what order.
    Guarantees SQL runs before any data-dependent agent, validates the chart plan
    against the sample data, and falls back to a safe deterministic chart if
    model-produced code fails (e.g., uses a missing column).

    Returns: {"trace", "sql", "df", "fig", "analysis"}
    """
    trace: Dict[str, Any] = {"supervisor_prompt": supervisor_prompt, "actions": []}
    supervisor_query = (
        f"User question: {user_question}\n\n"
        "Decide which subagent(s) to call and in what order. Return names one-per-line, e.g.:\n"
        "sql-generator-agent\nchart-generator-agent\ndata-analysis-agent\n\n"
        "Only return agent names (no extra prose)."
    )
    trace["actions"].append({"supervisor_query": supervisor_query})

    # convert to the chat input format used by compiled_supervisor.stream
    cc_msgs = to_chat_completions_input([{"role": "user", "content": supervisor_query}])

    # Stream supervisor to capture node order and snippets
    node_order: list[str] = []
    try:
        seen_ids = set()
        for _, events in compiled_supervisor.stream({"messages": cc_msgs}, stream_mode=["updates"]):
            if not events:
                continue
            node_name = tuple(events.keys())[0]
            msgs = events[node_name].get("messages", []) or []
            new_msgs = [m for m in msgs if m.id not in seen_ids]
            if not new_msgs:
                continue
            for m in new_msgs:
                seen_ids.add(m.id)
            if node_name not in node_order:
                node_order.append(node_name)
            # capture short preview of latest messages for audit
            snippets = []
            for m in new_msgs:
                content = getattr(m, "content", None)
                if content is None and hasattr(m, "text"):
                    content = m.text
                snippets.append(str(content)[:400])
            trace["actions"].append({"supervisor_node_update": {"node": node_name, "snippet": snippets}})
    except Exception as e:
        trace["actions"].append({"supervisor_stream_error": str(e)})

    # fallback order if supervisor didn't return anything clear
    if not node_order:
        node_order = ["sql-generator-agent", "data-analysis-agent", "chart-generator-agent"]
        trace["actions"].append({"supervisor_fallback_order": node_order})
    else:
        trace["actions"].append({"supervisor_decided_order": node_order})

    # prepare output containers
    generated_sql = ""
    sql_result = {"rows": [], "columns": [], "rowcount": 0}
    df = pd.DataFrame()
    sample_json = "[]"
    schema_str = ""
    analysis_output = None
    chart_code = None
    fig = None
    chart_choice = None

    # helper: robust conversion rows->DataFrame
    def _rows_to_dataframe(rows, columns):
        if rows is None:
            return pd.DataFrame()
        if isinstance(rows, list) and rows and isinstance(rows[0], dict):
            return pd.DataFrame(rows)
        try:
            df_try = pd.DataFrame(rows)
            if columns and isinstance(columns, list):
                if all(isinstance(c, dict) and "name" in c for c in columns):
                    col_names = [c["name"] for c in columns]
                else:
                    col_names = columns
                if df_try.shape[1] == len(col_names):
                    df_try.columns = col_names
            return df_try
        except Exception:
            try:
                serializable = [json.loads(json.dumps(r, default=str)) for r in rows]
                return pd.DataFrame(serializable)
            except Exception:
                return pd.DataFrame()

    # helper to ensure SQL generated & executed
    def ensure_sql():
        nonlocal generated_sql, sql_result, df, sample_json, schema_str
        if generated_sql:
            return
        try:
            sql_text = sql_agent_call(user_question)
            m = re.search(r"(?i)(select\b[\s\S]*)", sql_text or "")
            generated_sql = m.group(1).strip() if m else (sql_text or "")
            trace["actions"].append({"sql_generated_raw": sql_text, "generated_sql": generated_sql})
        except Exception as e:
            trace["actions"].append({"sql_agent_error": str(e)})
            generated_sql = ""
            return

        if generated_sql:
            try:
                sql_result = run_sql(generated_sql, max_rows=MAX_SQL_ROWS)
                rows = sql_result.get("rows", [])
                cols = sql_result.get("columns") or []
                df_local = _rows_to_dataframe(rows, cols)
                df_local = df_local.copy()
                df = df_local  # assign to outer-scope df
                schema_str = json.dumps({"columns": sql_result.get("columns")}, indent=2, default=str)
                sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str, indent=2)
                trace["actions"].append({"run_sql_success_rowcount": int(sql_result.get("rowcount", len(df)))})
            except Exception as e:
                trace["actions"].append({"run_sql_error": str(e)})
                sql_result = {"rows": [], "columns": [], "rowcount": 0}
                df = pd.DataFrame()
                sample_json = "[]"
                schema_str = ""

    # chart plan validator: given sample_json and plan ensure columns exist and adjust
    def _validate_and_repair_plan(plan_obj, sample_df):
        # internal helpers
        def _valid_col(col):
            return col is not None and isinstance(col, str) and col in list(sample_df.columns)

        # chart_type
        if plan_obj.get("chart_type") not in {"LINE", "BAR", "PIE", "HEATMAP"}:
            plan_obj["chart_type"] = "LINE" if "DATE" in sample_df.columns else "BAR"

        # x
        if not _valid_col(plan_obj.get("x")):
            if "DATE" in sample_df.columns:
                plan_obj["x"] = "DATE"
            else:
                non_numeric = [c for c in sample_df.columns if not pd.api.types.is_numeric_dtype(sample_df[c])]
                plan_obj["x"] = non_numeric[0] if non_numeric else (list(sample_df.columns)[0] if len(sample_df.columns) else None)

        # y
        y = [c for c in (plan_obj.get("y") or []) if c in sample_df.columns]
        if not y:
            numeric = [c for c in sample_df.columns if pd.api.types.is_numeric_dtype(sample_df[c]) and c != plan_obj["x"]]
            if numeric:
                y = numeric[:2]
            else:
                y = [c for c in sample_df.columns if c != plan_obj["x"]][:2]
        plan_obj["y"] = y

        # aggregation
        if plan_obj.get("aggregation") not in {"sum", "avg", "count", None}:
            plan_obj["aggregation"] = "sum"

        # date_construction
        dc = plan_obj.get("date_construction") or {}
        if dc.get("type") == "year_month":
            if not (_valid_col(dc.get("year_col")) and _valid_col(dc.get("month_col"))):
                dc = {"type": None}
        plan_obj["date_construction"] = dc
        return plan_obj

    # mapping of node names to actions (lower-casing match)
    for node in node_order:
        node_lower = node.lower()
        trace["actions"].append({"executing_node": node})
        if "sql" in node_lower:
            ensure_sql()
        elif "data" in node_lower:
            # ensure SQL to provide data context
            ensure_sql()
            try:
                analysis_output = data_analysis_agent_call(user_question, schema_str, sample_json)
                trace["actions"].append({"data_analysis_output_preview": str(analysis_output)[:400]})
            except Exception as e:
                trace["actions"].append({"data_analysis_error": str(e)})
                analysis_output = None
        elif "chart" in node_lower or "visual" in node_lower:
            ensure_sql()
            # Request chart plan + code via chart_agent_call (chart_agent_call returns code; here we still validate before exec)
            try:
                # If chart_agent_call returns code only, we still validate via sample_df by attempting to extract a plan
                # We'll call chart_agent_call and then try to detect a plan if it returns one; otherwise rely on fallback behavior below.
                chart_code = chart_agent_call(user_question, sample_json)
                trace["actions"].append({"raw_chart_code_snippet": (chart_code[:400] if chart_code else None)})
            except Exception as e:
                trace["actions"].append({"chart_agent_error": str(e)})
                chart_code = None

            # Attempt to compile & execute the returned code, but defend against missing x/y columns
            if chart_code:
                # Clean top-level import lines (we provide pd/px/go)
                cleaned_lines = [ln for ln in chart_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
                cleaned_code = "\n".join(cleaned_lines).strip()

                # Execution env
                local_env = {"pd": pd, "px": px, "go": go, "df": df.copy(), "__builtins__": {"len": len, "range": range, "min": min, "max": max, "sum": sum}}
                try:
                    compiled = compile(cleaned_code, "<chart_code>", "exec")
                    exec(compiled, local_env)
                    fig = local_env.get("fig")
                    trace["actions"].append({"chart_exec_success": bool(fig)})
                except Exception as e:
                    trace["actions"].append({"chart_exec_error": str(e)})
                    # If error indicates missing column names (common), attempt fallback deterministic code that chooses valid columns
                    err_str = str(e)
                    if "is not the name of a column" in err_str or "Expected one of" in err_str or "KeyError" in err_str:
                        try:
                            fb_code = fallback_chart_code(user_question, df.head(5), chart_choice=chart_choice or "BAR")
                            fb_lines = [ln for ln in fb_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
                            compiled_fb = compile("\n".join(fb_lines), "<fallback_chart>", "exec")
                            exec(compiled_fb, local_env)
                            fig = local_env.get("fig")
                            trace["actions"].append({"fallback_chart_exec_success": bool(fig)})
                        except Exception as e2:
                            trace["actions"].append({"fallback_chart_exec_error": str(e2)})
                            fig = None
                    else:
                        # non-column error: try fallback as well
                        try:
                            fb_code = fallback_chart_code(user_question, df.head(5), chart_choice=chart_choice or "BAR")
                            fb_lines = [ln for ln in fb_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
                            compiled_fb = compile("\n".join(fb_lines), "<fallback_chart>", "exec")
                            exec(compiled_fb, local_env)
                            fig = local_env.get("fig")
                            trace["actions"].append({"fallback_chart_exec_success_non_col_error": bool(fig)})
                        except Exception as e3:
                            trace["actions"].append({"fallback_chart_exec_error_non_col": str(e3)})
                            fig = None
            else:
                # No code from agent — use deterministic fallback
                try:
                    fb_code = fallback_chart_code(user_question, df.head(5), chart_choice=chart_choice or "BAR")
                    fb_lines = [ln for ln in fb_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
                    local_env = {"pd": pd, "px": px, "go": go, "df": df.copy(), "__builtins__": {"len": len, "range": range, "min": min, "max": max, "sum": sum}}
                    compiled_fb = compile("\n".join(fb_lines), "<fallback_chart>", "exec")
                    exec(compiled_fb, local_env)
                    fig = local_env.get("fig")
                    trace["actions"].append({"fallback_chart_exec_success_no_agent_code": bool(fig)})
                except Exception as e:
                    trace["actions"].append({"fallback_chart_exec_error_no_agent_code": str(e)})
                    fig = None
        else:
            trace["actions"].append({"unknown_node_ignored": node})

    # final return
    return {"trace": trace, "sql": generated_sql, "df": df, "fig": fig, "analysis": analysis_output}





# Example run
if __name__ == "__main__":
    q = "Show total SPENDS and total HH_GRPS for DOG species. Plot the values for both metrics in a bar chart for different brands."
    out = nl_to_chart_and_analysis_agentic(q)
    print("Trace:", json.dumps(out["trace"], indent=2))
    print("Data shape:", out["df"].shape if isinstance(out["df"], pd.DataFrame) else None)
    if out.get("analysis"):
        print("Data Analysis Output:\n", out["analysis"])


In [0]:
# %%writefile agent_complete_with_chart.py
import json
import re
from uuid import uuid4
from typing import Generator, List, Optional
from dataclasses import dataclass
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
from langgraph.graph import StateGraph, END
from langgraph.graph.state import CompiledStateGraph
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest, ResponsesAgentResponse, ResponsesAgentStreamEvent,
    output_to_responses_items_stream, to_chat_completions_input,
)

# Setup LLM and Databricks client
client = DatabricksFunctionClient()
set_uc_function_client(client)

@dataclass
class ToolCall:
    tool: str
    input: dict
    agent: str
    reasoning: str

@dataclass
class ToolResult:
    tool: str
    output: any
    success: bool
    error: Optional[str]
    metadata: dict

class ToolExecutor:
    def __init__(self, spark, llm, table_name: str, max_rows: int = 2000):
        self.spark = spark
        self.llm = llm
        self.table_name = table_name
        self.max_rows = max_rows
    
    def execute(self, tool_call: ToolCall) -> ToolResult:
        try:
            if tool_call.tool == "sql_validate":
                return self._validate_sql(tool_call.input)
            if tool_call.tool == "sql_execute":
                return self._execute_sql(tool_call.input)
            if tool_call.tool == "data_profile":
                return self._profile_data(tool_call.input)
            if tool_call.tool == "plot_render":
                return self._render_plot(tool_call.input)
            return ToolResult(tool=tool_call.tool, output=None, success=False, error="Unknown tool", metadata={})
        except Exception as e:
            return ToolResult(tool=tool_call.tool, output=None, success=False, error=str(e), metadata={})
    
    def _validate_sql(self, input_dict: dict) -> ToolResult:
        query = input_dict["query"]
        if "SELECT" not in query.upper():
            return ToolResult("sql_validate", None, False, "No SELECT in query", {})
        forbidden = ["DROP", "DELETE", "TRUNCATE", "ALTER"]
        if any(kw in query.upper() for kw in forbidden):
            return ToolResult("sql_validate", None, False, "Destructive statement forbidden", {})
        return ToolResult("sql_validate", {"valid": True}, True, None, {})
    
    def _execute_sql(self, input_dict: dict) -> ToolResult:
        query = input_dict["query"]
        val = self._validate_sql(input_dict)
        if not val.success:
            return ToolResult("sql_execute", None, False, val.error, {})
        df = self.spark.sql(query).limit(self.max_rows).toPandas()
        output = {"dataframe": df, "rows": json.loads(df.to_json(orient="records")), "columns": list(df.columns), "dtypes": {col:str(dtype) for col, dtype in df.dtypes.items()}}
        return ToolResult("sql_execute", output, True, None, {"rows_fetched": len(df)})
    
    def _profile_data(self, input_dict: dict) -> ToolResult:
        df = input_dict.get("dataframe")
        if df is None or df.empty:
            return ToolResult("data_profile", None, False, "Empty dataframe", {})
        profile = {
            "null_counts": df.isnull().sum().to_dict(),
            "unique_counts": {c: df[c].nunique() for c in df.columns},
            "shape": df.shape
        }
        return ToolResult("data_profile", profile, True, None, {})
    
    def _render_plot(self, input_dict: dict) -> ToolResult:
        code = input_dict.get("code", "")
        try:
            compile(code, "<chart_code>", "exec")
            return ToolResult("plot_render", {"compiled": True}, True, None, {})
        except Exception as e:
            return ToolResult("plot_render", None, False, str(e), {})

class CustomAgent:
    def __init__(self, name: str, llm, tool_executor: ToolExecutor):
        self.name = name
        self.llm = llm
        self.tool_executor = tool_executor
        self.max_iter = 3
    
    def call_llm(self, prompt: str) -> str:
        try:
            if hasattr(self.llm, "invoke"):
                out = self.llm.invoke(prompt)
            else:
                out = self.llm(prompt)
            if isinstance(out, str):
                return out
            if hasattr(out, "content"):
                return out.content
            if hasattr(out, "text"):
                return out.text
            return str(out)
        except Exception as e:
            print(f"LLM call error: {e}")
            return ""
    
    def think(self, state: dict) -> str:
        prompt = f"Agent: {self.name}\nQuestion: {state.get('question', '')}\nNext action?"
        return self.call_llm(prompt)
    
    def act(self, state: dict, thought: str) -> List[ToolResult]:
        return []
    
    def run(self, state: dict) -> dict:
        for _ in range(self.max_iter):
            thought = self.think(state)
            results = self.act(state, thought)
            if all(r.success for r in results):
                break
            state.setdefault("tool_results", []).extend(results)
        return state

class SQLGeneratorAgent(CustomAgent):
    def __init__(self, llm, tool_executor, table_name):
        super().__init__("SQLGeneratorAgent", llm, tool_executor)
        self.table_name = table_name

    def act(self, state: dict, thought: str) -> List[ToolResult]:
        question = state.get("question", "")
        prompt = f"""You are an expert SQL generator for Databricks Delta tables.
Available table: `{self.table_name}` with columns:
DATE, ZONE, REGION, COUNTRY, RETAIL_CHANNEL, RETAILER_NAME, MANUFACTURER, PRODUCT_FAMILY, SPECIES, BRAND, SUB_BRAND, SKU_NAME, MARKETING_CHANNEL, CAMPAIGN_NAME, METRIC, VALUE
Rules:
- Return a single VALID Databricks SQL SELECT statement, no explanation, no markdown fences.
- Use uppercase for SQL keywords.
- Use ISO date format 'YYYY-MM-DD'.
- Always include an ORDER BY.
- Do NOT include destructive statements.
User question: {question}"""
        raw_sql = self.call_llm(prompt)
        sql_query = re.search(r"(?i)(select\b[\s\S]*)", raw_sql)
        sql_query = sql_query.group(1).strip() if sql_query else raw_sql.strip()
        val = self.tool_executor.execute(ToolCall("sql_validate", {"query": sql_query}, self.name, thought))
        results = [val]
        if val.success:
            exec_result = self.tool_executor.execute(ToolCall("sql_execute", {"query": sql_query}, self.name, "Execute query"))
            results.append(exec_result)
            if exec_result.success:
                state["generated_sql"] = sql_query
                state["dataframe"] = exec_result.output["dataframe"]
                state["schema"] = {"columns": exec_result.output["columns"], "dtypes": exec_result.output["dtypes"]}
        return results

class DataAnalysisAgent(CustomAgent):
    def __init__(self, llm, tool_executor):
        super().__init__("DataAnalysisAgent", llm, tool_executor)

    def act(self, state: dict, thought: str) -> List[ToolResult]:
        df = state.get("dataframe")
        if df is None or df.empty:
            return []
        profile_result = self.tool_executor.execute(ToolCall("data_profile", {"dataframe": df}, self.name, "Profile data"))
        results = [profile_result]
        question = state.get("question", "")
        schema_str = json.dumps(state.get("schema", {}))
        sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str)
        analysis_prompt = f"""You are a data analysis expert. Provide key insights and recommendations:
Schema: {schema_str}
Sample data: {sample_json}
User question: {question}
Format: Plain text only."""
        analysis_text = self.call_llm(analysis_prompt)
        state["analysis_text"] = analysis_text
        results.append(ToolResult("text_analysis", {"text": analysis_text}, True, None, {}))
        return results

class ChartGeneratorAgent(CustomAgent):
    def __init__(self, llm, tool_executor):
        super().__init__("ChartGeneratorAgent", llm, tool_executor)

    def act(self, state: dict, thought: str) -> List[ToolResult]:
        df = state.get("dataframe")
        if df is None or df.empty:
            return []

        # profile the dataframe (keeps existing behavior)
        profile_result = self.tool_executor.execute(ToolCall("data_profile", {"dataframe": df}, self.name, "Profile data for chart"))
        results = [profile_result]

        question = state.get("question", "")
        sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str)

        # Strict plan prompt: force using exact sample columns and forbid creating new data
        plan_prompt = (
            "You are a visualization planner. Return ONLY a single JSON object (no prose). "
            "Use exact column names from the sample (case-sensitive). DO NOT invent or fabricate columns. "
            "DO NOT create or return a DataFrame. The chart must be based on the existing dataframe variable `df`.\n\n"
            "Return JSON with keys: chart_type (LINE|BAR|PIE|HEATMAP), x (column or null), y (list of 1-2 columns), "
            "aggregation (sum|avg|count|null), title (short string).\n\n"
            f"Sample data: {sample_json}\n"
            f"Question: {question}\n"
        )

        # Ask LLM for plan
        plan_json = self.call_llm(plan_prompt)

        # Parse plan safely and repair using actual df columns if needed
        try:
            cleaned_json_text = re.sub(r"^```(?:json)?|```$", "", plan_json, flags=re.IGNORECASE).strip()
            plan = json.loads(cleaned_json_text)
        except Exception:
            # conservative default plan derived from actual df columns (never fabricate)
            plan = {"chart_type": "LINE", "x": None, "y": list(df.columns)[:2], "aggregation": "sum", "title": question[:80]}

        # Repair/validate plan to ensure referenced columns actually exist in df
        def _valid_col(col):
            return col is not None and isinstance(col, str) and col in list(df.columns)

        # Normalize chart_type
        if plan.get("chart_type") not in {"LINE", "BAR", "PIE", "HEATMAP"}:
            plan["chart_type"] = "LINE" if "DATE" in df.columns else "BAR"

        # Validate x
        if not _valid_col(plan.get("x")):
            if "DATE" in df.columns:
                plan["x"] = "DATE"
            else:
                non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
                plan["x"] = non_numeric[0] if non_numeric else (list(df.columns)[0] if len(df.columns) else None)

        # Validate y (must be actual columns)
        y_candidates = [c for c in (plan.get("y") or []) if c in df.columns]
        if not y_candidates:
            numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != plan.get("x")]
            y_candidates = numeric[:2] if numeric else [c for c in df.columns if c != plan.get("x")][:2]
        plan["y"] = y_candidates

        # Validate aggregation
        if plan.get("aggregation") not in {"sum", "avg", "count", None}:
            plan["aggregation"] = "sum"

        # Build a strict code prompt that enforces use of `df` and forbids creating new data
        code_prompt = (
            "You are a Python/Plotly code generator. Produce ONLY executable Python code (no prose, no fences). "
            "IMPORTANT RULES:\n"
            "- The code MUST use the existing pandas DataFrame variable named `df` (this is the full query result).\n"
            "- DO NOT create a new DataFrame or hardcode sample values.\n"
            "- DO NOT reference columns that are not present in `df`.\n"
            "- The code MUST assign a Plotly figure to variable `fig` and call `fig.show()` as the last statement.\n"
            "- Do NOT include import statements for pandas/plotly (pd/px/go are provided by runtime).\n"
            "- Use defensive checks like `if 'COL' in df.columns` before using a column.\n\n"
            f"Plan JSON: {json.dumps(plan)}\n"
            f"Question: {question}\n"
            f"Sample columns: {list(df.columns)}\n"
        )

        # Request code from LLM
        code = self.call_llm(code_prompt)

        # Strip code fences and top-level import lines (we provide pd/px/go)
        code = re.sub(r"^```(?:python)?\s*", "", code, flags=re.IGNORECASE).strip()
        code = re.sub(r"\s*```\s*$", "", code, flags=re.IGNORECASE)
        code_lines = [ln for ln in code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
        cleaned_code = "\n".join(code_lines).strip()

        # Quick static checks: must reference df, must define fig, must call fig.show
        if ("df" not in cleaned_code) or ("fig" not in cleaned_code) or ("fig.show" not in cleaned_code):
            # do not accept code that creates its own DataFrame or doesn't operate on df
            validation = ToolResult(success=False, output="Chart code missing required references (df/fig/fig.show).")
            results.append(validation)
            return results

        # Attempt to compile to catch syntax errors before remote validation
        try:
            compile(cleaned_code, "<chart_code>", "exec")
        except Exception as e:
            validation = ToolResult(success=False, output=f"Chart code compilation error: {e}")
            results.append(validation)
            return results

        # Ask the tool_executor to validate/render the chart (existing behavior)
        validation = self.tool_executor.execute(ToolCall("plot_render", {"code": cleaned_code, "dataframe": df}, self.name, "Validate chart code"))
        results.append(validation)

        if validation.success:
            # store the cleaned code (which uses real df) into state for downstream execution
            state["chart_code"] = cleaned_code
        else:
            # if validation failed, do not set chart_code (strict mode, no fallback)
            state["chart_code"] = ""

        return results


def create_custom_agent_langgraph(llm, tool_executor, table_name="demo.retail_media"):
    sql_agent = SQLGeneratorAgent(llm, tool_executor, table_name)
    analysis_agent = DataAnalysisAgent(llm, tool_executor)
    chart_agent = ChartGeneratorAgent(llm, tool_executor)

    def supervisor_node(state: dict) -> dict:
        messages = state.get("messages", [])
        if not messages:
            return state
        last_msg = messages[-1].get("content", "")
        state["question"] = last_msg
        return state

    def sql_node(state: dict) -> dict:
        state["tool_results"] = []
        return sql_agent.run(state)

    def analysis_node(state: dict) -> dict:
        return analysis_agent.run(state)

    def chart_node(state: dict) -> dict:
        return chart_agent.run(state)

    graph = StateGraph(dict)
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("sql_agent", sql_node)
    graph.add_node("analysis_agent", analysis_node)
    graph.add_node("chart_agent", chart_node)

    graph.add_edge("supervisor", "sql_agent")
    graph.add_edge("sql_agent", "analysis_agent")
    graph.add_edge("analysis_agent", "chart_agent")
    graph.add_edge("chart_agent", END)

    graph.set_entry_point("supervisor")
    return graph.compile()

class SimpleAgent:
    def __init__(self, graph: CompiledStateGraph):
        self.graph = graph
    
    def run(self, question: str) -> dict:
        input_state = {
            "messages": [{"content": question}],
            "question": question
        }
        final_state = None
        for step in self.graph.stream(input_state):
            final_state = step
        return final_state or input_state

# Initialization
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
TABLE_NAME = "demo.retail_media"
MAX_SQL_ROWS = 2000

print("Setting up LLM and executor...")
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
tool_executor = ToolExecutor(spark, llm, TABLE_NAME, MAX_SQL_ROWS)
compiled_graph = create_custom_agent_langgraph(llm, tool_executor, TABLE_NAME)
agent = SimpleAgent(compiled_graph)
print("Setup complete. Ready to run.")

# To run:
# result = agent.run("Your NL question here")
# print(result.get("analysis_text"))
# print(result.get("chart_code"))
# result['dataframe'].head()  # etc.


In [0]:
prompt="Show total SPENDS and total HH_GRPS for DOG species. Plot barchart barndwise."
result = agent.run(prompt)
print(result)

In [0]:
def run_agent_and_display_chart(prompt: str):
    print("\n" + "="*70)
    print("📊 CHART GENERATION")
    print("="*70)
    print(f"❓ Prompt: {prompt}\n")

    print("🔄 Running agent pipeline...")
    print("   - SQL Generation")
    print("   - Data Analysis")
    print("   - Chart Generation\n")

    try:
        # Run agent pipeline
        result = agent.run(prompt)
         
        if result:
            print("✅ Agent execution completed!\n")
            print(result)

            # Extract results
            chart_code = result.get('chart_code')
            dataframe = result.get('dataframe')
            generated_sql = result.get('generated_sql')
            analysis_text = result.get('analysis_text')

            # Display SQL
            print("📝 GENERATED SQL:")
            print("-" * 70)
            print(generated_sql if generated_sql else "N/A")
            print("-" * 70 + "\n")

            # Display data analysis
            print("📊 DATA ANALYSIS:")
            print("-" * 70)
            print(analysis_text if analysis_text else "N/A")
            print("-" * 70 + "\n")

            # Execute and display chart
            if chart_code and dataframe is not None and not dataframe.empty:
                print("📉 GENERATING CHART...")
                print("-" * 70)

                try:
                    # Clean chart code (remove any import statements)
                    cleaned_lines = [ln for ln in chart_code.split('\n')
                                     if not re.match(r'^\s*(import|from)\s+', ln)]
                    cleaned_code = '\n'.join(cleaned_lines)

                    # Prepare safe environment for exec
                    safe_globals = {
                        'pd': pd,
                        'px': px,
                        'go': go,
                        'df': dataframe.copy(),
                    }

                    # Execute the visualization code
                    exec(cleaned_code, safe_globals, safe_globals)

                    # Retrieve the figure object from executed code
                    fig = safe_globals.get('fig')

                    if fig:
                        # Display the Plotly chart inline
                        fig.show()
                        print("\n✅ Chart displayed successfully!\n")

                        # Optional: Display data summary
                        print("📋 DATA SUMMARY:")
                        print("-" * 70)
                        print(f"Rows: {len(dataframe)}")
                        print(f"Columns: {list(dataframe.columns)}")
                        print(f"Shape: {dataframe.shape}")
                        print("-" * 70)
                    else:
                        print("❌ No figure created from code")

                except Exception as e:
                    print(f"❌ Error executing chart code: {e}")
                    print("\nChart Code:")
                    print(chart_code)
            else:
                print("❌ No chart code or dataframe available")
        else:
            print("❌ No results from agent")

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()


In [0]:
run_agent_and_display_chart("Show total SPENDS and total HH_GRPS for DOG species. Plot barchart barndwise.")


In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame with columns 'species', 'brand', 'SPENDS', 'HH_GRPS'
dog_data = data[data['species'] == 'DOG']

# Calculate totals by brand
brand_totals = dog_data.groupby('brand')[['SPENDS', 'HH_GRPS']].sum().reset_index()

# Plotting
plt.figure(figsize=(10, 6))
bar_width = 0.4
x = range(len(brand_totals['brand']))
plt.bar([i - bar_width/2 for i in x], brand_totals['SPENDS'], width=bar_width, label='SPENDS')
plt.bar([i + bar_width/2 for i in x], brand_totals['HH_GRPS'], width=bar_width, label='HH_GRPS')
plt.xticks(x, brand_totals['brand'], rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# Example test query
user_question = "Show total SPENDS and total HH_GRPS for DOG species. Plot the values for both metrics in a time series chart month and year wise."

# Run the agent with the query
result = agent.run(user_question)

# Display the outputs
print("Generated SQL:\n", result.get("generated_sql", "No SQL generated"))
print("\nData Analysis:\n", result.get("analysis_text", "No analysis"))
print("\nChart Code:\n", result.get("chart_code", "No chart code"))

# Display a preview of the resulting data
df = result.get("dataframe")
if df is not None:
    print("\nData preview:")
    display(df.head())
else:
    print("No data returned from query.")


In [0]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "BRAND": ["Brand1", "Brand2", "Brand1", "Brand2"],
    "TOTAL_SPENDS": [100, 200, 150, 250],
    "DOG_SPECIES": ["Species1", "Species1", "Species2", "Species2"]
})

if 'BRAND' in df.columns and 'TOTAL_SPENDS' in df.columns:
    # if plan['aggregation'] is not None:
    df = df.groupby('BRAND')[plan['y'][0]].sum().reset_index()
    # else:
    #     df = df.groupby('BRAND')[plan['y'][0]].sum().reset_index() if 'DOG_SPECIES' in df.columns else df
    
    fig = px.bar(df, x='BRAND', y=plan['y'][0], title=plan['title'])
    fig.show()

In [0]:
# Cell 3
def to_json_friendly(v: Any):
    """Convert common non-JSON types (dates, timestamps, numpy, pandas) to natives/strings."""
    try:
        if v is None or isinstance(v, (str, bool, int, float)):
            if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
                return None
            return v
        if isinstance(v, (datetime.datetime, datetime.date, pd.Timestamp)):
            return v.isoformat()
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (np.floating,)):
            return float(v)
        if isinstance(v, (np.bool_,)):
            return bool(v)
        if isinstance(v, decimal.Decimal):
            return float(v)
        if isinstance(v, pd.Series):
            return [to_json_friendly(x) for x in v.head().tolist()]
        if isinstance(v, pd.DataFrame):
            return {
                "type": "pandas.DataFrame",
                "columns": [str(c) for c in v.columns.tolist()],
                "rows_sampled": int(len(v)),
                "sample_head": [
                    {col: to_json_friendly(row[col]) for col in v.columns}
                    for _, row in v.head(5).iterrows()
                ]
            }
        if isinstance(v, dict):
            return {str(k): to_json_friendly(val) for k, val in v.items()}
        if isinstance(v, (list, tuple, set)):
            return [to_json_friendly(x) for x in list(v)]
        # fallback: try json dumps else str
        try:
            json.dumps(v)
            return v
        except Exception:
            return str(v)
    except Exception:
        return str(v)

def make_serializable(obj: Any):
    """Recursively make an object JSON-friendly."""
    return to_json_friendly(obj)

from langchain.schema import HumanMessage

def call_llm(prompt, system=None, max_tokens=None):
    # If you have a system prompt, include it as a SystemMessage
    messages = []
    if system:
        from langchain.schema import SystemMessage
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm(messages)
    # Extract content robustly
    try:
        return resp.content
    except AttributeError:
        return resp


In [0]:
# Cell 4
def supervisor_loop(user_query: str, max_steps: int = 6):
    """
    Minimal loop: DataAnalysis -> SQL -> Visualization.
    Each agent is LLM-driven. Loop ends after visualization or if agent returns "END".
    Returns final_state dict with serializable summaries and full pandas sample for viz.
    """
    state = {
        "user_query": user_query,
        "agent_results": {},   # will contain serializable summaries + full pandas for viz
        "messages": []
    }

    # Step 1: Data Analysis (LLM)
    data_prompt = f"""
You are a data analyst assistant. User asked: {user_query}

1. Inspect available table: {TARGET_TABLE}.
2. Return a JSON object with keys:
   - data_ready: true|false
   - missing_columns: [ ... ]  (if any)
   - suggested_transforms: [ ... ] (e.g., combine sales_year+sales_month -> sales_date)
   - sample_summary: {{ columns: [...], rows_sampled: N, sample_head: [...] }}   # only if you can query schema/sample
Return ONLY the JSON. Use the actual table schema and a small sample if possible.
"""
    # get schema + small sample if EXECUTE_SQL
    schema_info = None
    sample_pdf = None
    if EXECUTE_SQL:
        try:
            schema = [{"name": f.name, "type": f.dataType.simpleString()} for f in spark.table(TARGET_TABLE).schema.fields]
            sample_pdf = spark.table(TARGET_TABLE).limit(1000).toPandas()
            schema_info = {"schema": schema}
        except Exception as e:
            schema_info = {"error": str(e)}
    # embed serializable context
    context = {"table": TARGET_TABLE, "schema_info": make_serializable(schema_info)}
    prompt = data_prompt + "\n\nContext: " + json.dumps(context, indent=2)
    resp = call_llm(prompt)
    # parse JSON from response
    try:
        jstart = resp.find("{")
        jend = resp.rfind("}")+1
        analysis_json = json.loads(resp[jstart:jend])
    except Exception:
        # fallback: mark not ready
        analysis_json = {"data_ready": False, "missing_columns": [], "suggested_transforms": [], "sample_summary": {}}
    # store analysis summary (serializable)
    state["agent_results"]["data_analysis"] = make_serializable(analysis_json)
    state["messages"].append({"agent": "data_analysis", "text": resp})

    if not analysis_json.get("data_ready", False):
        # end early — return structured suggestion to user
        return state

    # Step 2: SQL generation (LLM)
    sql_prompt = f"""
You are a SQL generator. User query: {user_query}
Data analysis output: {json.dumps(make_serializable(analysis_json), indent=2)}
Table: {TARGET_TABLE}
Rules:
 - Return only one SQL statement.
 - If sales_year and sales_month exist, prefer MAKE_DATE(sales_year, sales_month, 1) as sales_date.
 - Ensure query returns: sales_date (or equivalent), product_category (if requested), aggregated measures (sum of amount) by date.
 - Keep query safe for Spark SQL / Delta Lake.
Return ONLY the SQL (no explanation).
"""
    sql_resp = call_llm(sql_prompt)
    # try to extract SQL (from first "SELECT")
    sel = sql_resp.upper().find("SELECT")
    sql_text = sql_resp if sel == -1 else sql_resp[sel:]
    state["agent_results"]["sql_text"] = sql_text
    state["messages"].append({"agent": "sql_generation", "text": sql_resp})

    # Execute SQL (optional) and keep sample for viz
    pdf = None
    if EXECUTE_SQL:
        try:
            df = spark.sql(sql_text)
            pdf = df.limit(SQL_LIMIT).toPandas()
            state["agent_results"]["sql_df_sample"] = pdf    # keep full for viz
            state["agent_results"]["sql_df_summary"] = make_serializable({
                "columns": pdf.columns.tolist(),
                "rows_sampled": len(pdf),
                "sample_head": pdf.head(5).to_dict(orient="records")
            })
            state["messages"].append({"agent": "sql_execution", "text": f"Executed SQL, fetched {len(pdf)} rows (sample)."})
        except Exception as e:
            state["agent_results"]["sql_error"] = str(e)
            state["messages"].append({"agent": "sql_execution", "text": f"SQL execution failed: {e}"})
            return state

    # Step 3: Visualization (LLM-driven plan + local rendering)
    viz_prompt = f"""
You are a visualization assistant. Input: user query: {user_query}
SQL summary: {json.dumps(make_serializable(state['agent_results'].get('sql_df_summary', {})), indent=2)}
Task: Decide the best chart type and output a minimal Python plotting snippet using matplotlib or pandas that:
 - Loads a pandas DataFrame named `pdf` (already provided in notebook)
 - Draws the chart (line/bar/area) showing trends
 - Sets axis labels and legend
Return JSON with keys:
 - chart_type: "line"|"bar"|"table"
 - x: "<column>"
 - y: ["<col1>", ...]
 - code: "<python code block as a string>"
Return ONLY JSON.
"""
    viz_resp = call_llm(viz_prompt)
    try:
        s = viz_resp.find("{"); e = viz_resp.rfind("}")+1
        viz_json = json.loads(viz_resp[s:e])
    except Exception:
        # fallback: simple plan
        viz_json = {"chart_type": "table", "x": None, "y": [], "code": "print('No viz plan')"}
    state["agent_results"]["viz_plan"] = make_serializable(viz_json)
    state["messages"].append({"agent": "viz_planner", "text": viz_resp})

    # Execute plotting code safely: the code expects a pandas df variable named `pdf`
    if pdf is not None and viz_json.get("code"):
        # execute in a restricted locals context where pdf exists
        local_ns = {"pdf": pdf, "plt": plt, "pd": pd}
        try:
            exec(viz_json["code"], {}, local_ns)
            state["messages"].append({"agent": "viz_execution", "text": "Plot executed."})
            state["agent_results"]["final_answer"] = "Visualization rendered in notebook."
        except Exception as e:
            state["messages"].append({"agent": "viz_execution", "text": f"Plot execution failed: {e}"})
            state["agent_results"]["final_answer"] = f"Visualization failed: {e}"
    else:
        state["agent_results"]["final_answer"] = "No data for visualization."

    return state


In [0]:
# Cell 5 - Demo: change query as needed
query = "Show sales trends by product category over the last 2 years with a line chart"
state = supervisor_loop(query)
# print compact summary
print("=== Agent results summary ===")
print(json.dumps(make_serializable(state["agent_results"]), indent=2)[:4000])
print("\n=== Messages (last) ===")
for m in state["messages"][-6:]:
    print(m)


In [0]:
# Cell 3
class MultiAgentState(TypedDict):
    messages: List[Dict[str, Any]]      # {"role": "user"|"assistant", "content": "..."}
    user_query: str
    current_agent: str
    agent_results: Dict[str, Any]
    next_agent: str
    final_answer: str

def append_msg(state: MultiAgentState, role: str, content: str):
    state["messages"].append({"role": role, "content": content})

def extract_text_from_llm_response(resp) -> str:
    # Robust extraction for ChatDatabricks-style responses (adapt if your runtime differs)
    try:
        if isinstance(resp, list) and len(resp) > 0:
            first = resp[0]
            if hasattr(first, "content"):
                return first.content
            if isinstance(first, dict) and "content" in first:
                return first["content"]
            return str(first)
        if hasattr(resp, "content"):
            return resp.content
        return str(resp)
    except Exception:
        return str(resp)

# Decorator to register agent functions in a registry for clean graph building
_AGENT_REGISTRY: Dict[str, Callable[[MultiAgentState], MultiAgentState]] = {}

def register_agent(name: str):
    def decorator(func: Callable[[MultiAgentState], MultiAgentState]):
        _AGENT_REGISTRY[name] = func
        @wraps(func)
        def wrapper(state: MultiAgentState):
            return func(state)
        return wrapper
    return decorator


In [0]:
# Cell 4 - Tools (these are normal Python functions usable inside agent tasks)
from typing import Dict, Any, List
def tool_introspect_schema(table_name: str) -> Dict[str, Any]:
    """Return simple schema list (name, type) for given table or error."""
    try:
        df = spark.table(table_name)
        schema = [{"name": f.name, "type": f.dataType.simpleString()} for f in df.schema.fields]
        return {"ok": True, "schema": schema}
    except Exception as e:
        return {"ok": False, "error": str(e)}

def tool_run_sql(sql_text: str, limit: int = 10000) -> Dict[str, Any]:
    """Execute SQL and return a pandas sample or error. Runs on Spark cluster."""
    try:
        df = spark.sql(sql_text)
        pdf = df.limit(limit).toPandas()
        return {"ok": True, "dataframe": pdf}
    except Exception as e:
        return {"ok": False, "error": str(e)}

def tool_plot_dataframe(pdf, x_col: str, y_cols: List[str], title: str = None):
    """Simple plotting tool that returns matplotlib figure shown in notebook."""
    try:
        pdf[x_col] = pdf[x_col].astype("datetime64[ns]") if pdf[x_col].dtype == object or "datetime" not in str(pdf[x_col].dtype) else pdf[x_col]
    except Exception:
        # attempt best-effort conversion
        try:
            pdf[x_col] = pd.to_datetime(pdf[x_col])
        except Exception:
            pass

    plt.figure(figsize=(12,6))
    for col in y_cols:
        if col in pdf.columns:
            grouped = pdf.groupby(x_col)[col].sum().reset_index()
            grouped = grouped.sort_values(x_col)
            plt.plot(grouped[x_col], grouped[col], marker='o', label=col)
    plt.xlabel(x_col)
    plt.ylabel(", ".join(y_cols))
    if title:
        plt.title(title)
    plt.legend(); plt.grid(True); plt.tight_layout()
    display(plt.gcf())
    plt.close()
    return {"ok": True}
# Safe serialization helper (add to Cell 4 / helpers area)
import pandas as _pd

def safe_serialize_agent_results(agent_results: dict, sample_rows: int = 5) -> dict:
    """
    Convert agent_results into a JSON-serializable dict:
      - For pandas.DataFrame or Spark DataFrame objects it replaces them with a compact summary
        containing columns, row_count (if available), and sample_head (first N rows as list-of-dicts).
      - Leaves other simple types untouched.
    """
    serializable = {}
    for k, v in agent_results.items():
        # pandas DataFrame
        if isinstance(v, _pd.DataFrame):
            try:
                serializable[k] = {
                    "type": "pandas.DataFrame",
                    "columns": v.columns.tolist(),
                    "rows_sampled": len(v),
                    "sample_head": v.head(sample_rows).to_dict(orient="records")
                }
            except Exception as e:
                serializable[k] = {"type": "pandas.DataFrame", "error": str(e)}
        # sometimes we stored a dict containing a dataframe under a nested key (e.g., {"dataframe": df})
        elif isinstance(v, dict) and any(isinstance(x, _pd.DataFrame) for x in v.values()):
            serializable[k] = {}
            for subk, subv in v.items():
                if isinstance(subv, _pd.DataFrame):
                    try:
                        serializable[k][subk] = {
                            "type": "pandas.DataFrame",
                            "columns": subv.columns.tolist(),
                            "rows_sampled": len(subv),
                            "sample_head": subv.head(sample_rows).to_dict(orient="records")
                        }
                    except Exception as e:
                        serializable[k][subk] = {"type": "pandas.DataFrame", "error": str(e)}
                else:
                    # simple sub-value; include as-is if JSON-serializable
                    try:
                        json.dumps(subv)   # quick test
                        serializable[k][subk] = subv
                    except Exception:
                        serializable[k][subk] = str(subv)
        else:
            # Try to JSON-serialize value directly, otherwise convert to string
            try:
                json.dumps(v)
                serializable[k] = v
            except Exception:
                serializable[k] = str(v)
    return serializable



In [0]:
# Cell 5 - Data Analysis Agent
@register_agent("data_analysis_agent")
def data_analysis_agent(state: dict) -> dict:
    """
    Patched Data Analysis Agent:
    - saves a serializable sample summary in state['agent_results']['sample_pdf_summary']
    - preserves full pandas sample in state['agent_results']['sample_pdf'] for visualization
    """
    query = state["user_query"]
    append_msg(state, "assistant", f"🔍 Data Analysis Agent received the query: {query}")

    # schema introspection (if enabled)
    schema_res = tool_introspect_schema(TARGET_TABLE) if EXECUTE_SQL else {"ok": False, "error": "EXECUTE_SQL=False"}
    if schema_res.get("ok"):
        append_msg(state, "assistant", f"✅ Found schema for {TARGET_TABLE}: {json.dumps(schema_res['schema'])[:1000]}")
    else:
        append_msg(state, "assistant", f"⚠️ Schema introspection failed or disabled: {schema_res.get('error')}")

    # heuristics to detect required columns
    lower_q = query.lower()
    need_time = any(k in lower_q for k in ["trend", "over time", "by month", "monthly", "daily", "year", "quarter"])
    required = []
    if need_time:
        required.extend(["sales_year", "sales_month"])
    if "product" in lower_q or "category" in lower_q:
        required.append("product_category")

    # check missing columns against schema
    available_cols = [c["name"] for c in schema_res.get("schema", [])] if schema_res.get("schema") else []
    missing_columns = [c for c in required if c not in available_cols]

    # sample a small dataframe for validation and store summary
    sample_info = {}
    if schema_res.get("schema"):
        try:
            sample_pdf = spark.table(TARGET_TABLE).limit(1000).toPandas()
            sample_info["rows_sampled"] = len(sample_pdf)
            sample_info["columns"] = sample_pdf.columns.tolist()
            sample_info["sample_head"] = sample_pdf.head(5).to_dict(orient="records")
            # store full small sample for downstream use
            state["agent_results"]["sample_pdf"] = sample_pdf.head(200)
            # store compact serializable summary
            state["agent_results"]["sample_pdf_summary"] = {
                "columns": sample_pdf.columns.tolist(),
                "rows_sampled": len(sample_pdf),
                "sample_head": sample_pdf.head(5).to_dict(orient="records")
            }
            append_msg(state, "assistant", f"✅ Sample fetched (rows: {sample_info['rows_sampled']})")
        except Exception as e:
            append_msg(state, "assistant", f"⚠️ Sampling failed: {e}")

    # readiness and suggestions
    readiness = {
        "need_time": need_time,
        "required_columns": required,
        "missing_columns": missing_columns,
        "available_columns": available_cols,
        "sample_info": sample_info
    }
    if missing_columns:
        append_msg(state, "assistant", "❗ Missing columns detected: " + ", ".join(missing_columns))
        state["agent_results"]["data_ready"] = False
    else:
        append_msg(state, "assistant", "✅ Data appears to contain required columns for requested analysis.")
        state["agent_results"]["data_ready"] = True

    state["agent_results"]["data_analysis_agent"] = readiness
    state["current_agent"] = "data_analysis_agent"
    state["next_agent"] = "sql_agent" if state["agent_results"].get("data_ready", False) else "supervisor"
    return state


In [0]:
# Cell 6 - SQL Agent
@register_agent("sql_agent")
def sql_agent(state: dict) -> dict:
    append_msg(state, "assistant", "🧠 SQL Agent starting SQL generation/execution.")
    query = state["user_query"]
    analysis = state["agent_results"].get("data_analysis_agent", {})
    # use safe-serialized agent_results for prompt (avoid direct DFS objects)
    serial_agent_results = safe_serialize_agent_results(state.get("agent_results", {}))
    schema_res = tool_introspect_schema(TARGET_TABLE) if EXECUTE_SQL else {"ok": False}

    prompt = (
        "You are a SQL expert for a Databricks Delta table. Output ONLY the SQL (no explanation).\n\n"
        f"User query: {query}\n\n"
        f"Data analysis summary: {json.dumps(analysis)}\n\n"
        f"Agent results summary (safe): {json.dumps(serial_agent_results, indent=2)[:2000]}\n\n"
        f"Target table: {TARGET_TABLE}\n"
        f"Schema: {json.dumps(schema_res.get('schema', []), indent=2)[:3000]}\n\n"
        "If the user asks for trends over time, return a SQL that aggregates by a date column."
    )

    sql_text = None
    if USE_LLM_SQL_AGENT and llm is not None:
        try:
            resp = llm([HumanMessage(content=prompt)])
            sql_text = extract_text_from_llm_response(resp).strip()
            # attempt to extract SQL by finding first SELECT
            sel = sql_text.upper().find("SELECT")
            if sel != -1:
                sql_text = sql_text[sel:]
        except Exception as e:
            append_msg(state, "assistant", f"⚠️ LLM SQL generation failed: {e}; falling back to template.")

    if not sql_text:
        sql_text = f"""
SELECT 
  MAKE_DATE(sales_year, sales_month, 1) as sales_date,
  product_category,
  SUM(amount) as total_sales
FROM {TARGET_TABLE}
WHERE sales_year >= YEAR(CURRENT_DATE()) - 2
GROUP BY sales_date, product_category
ORDER BY sales_date
""".strip()
        append_msg(state, "assistant", "ℹ️ Using fallback SQL template (LLM unavailable or failed).")

    append_msg(state, "assistant", "🗃️ SQL generated:\n" + sql_text)
    state["agent_results"]["sql_query"] = sql_text
    state["current_agent"] = "sql_agent"

    # Execute SQL with retries and store both full pandas sample and serializable summary
    if EXECUTE_SQL:
        last_err = None
        for attempt in range(SQL_RETRY_COUNT + 1):
            exec_res = tool_run_sql(sql_text, limit=10000)
            if exec_res.get("ok"):
                pdf = exec_res["dataframe"]
                # store full sample for visualization
                state["agent_results"]["sql_df_sample"] = pdf
                # also store serializable summary for prompts/logging
                state["agent_results"]["sql_df_sample_summary"] = {
                    "columns": pdf.columns.tolist(),
                    "rows_sampled": len(pdf),
                    "sample_head": pdf.head(5).to_dict(orient="records")
                }
                append_msg(state, "assistant", f"✅ SQL executed successfully; sample rows: {len(pdf)}")
                last_err = None
                break
            else:
                last_err = exec_res.get("error")
                append_msg(state, "assistant", f"⚠️ SQL execution attempt {attempt+1} failed: {last_err}")
                time.sleep(SQL_RETRY_DELAY_SECONDS)
        if last_err:
            state["agent_results"]["sql_execution_error"] = str(last_err)
            append_msg(state, "assistant", "❌ SQL execution failed after retries; returning to supervisor.")
            state["next_agent"] = "supervisor"
            return state

    state["next_agent"] = "visualization_agent"
    return state


In [0]:
# Cell 7 - Visualization Agent
@register_agent("visualization_agent")
def visualization_agent(state: MultiAgentState) -> MultiAgentState:
    """
    Tasks:
     1) Inspect dataframe returned by SQL agent and decide chart type, axes, aggregation frequency.
     2) Render visualization using plotting tool.
    """
    append_msg(state, "assistant", "📈 Visualization Agent starting.")
    pdf = state["agent_results"].get("sql_df_sample")
    if pdf is None or pdf.empty:
        append_msg(state, "assistant", "⚠️ No dataframe available for visualization.")
        state["final_answer"] = "Visualization skipped: no data."
        state["next_agent"] = "supervisor"
        return state

    # Automatic chart selection heuristics:
    # If there's a date-like column and a numeric column -> line chart grouped by category if present
    cols = list(pdf.columns)
    date_cols = [c for c in cols if "date" in c.lower() or "dt" in c.lower() or "time" in c.lower()]
    num_cols = [c for c in cols if str(pdf[c].dtype).startswith(("int","float","double")) or c.lower().startswith("total") or "amount" in c.lower()]
    cat_cols = [c for c in cols if c not in date_cols + num_cols]

    chart = {"type": "table", "x": None, "y": None, "group_by": None}

    if date_cols and num_cols:
        chart["type"] = "line"
        chart["x"] = date_cols[0]
        chart["y"] = [num_cols[0]]
        chart["group_by"] = cat_cols[0] if cat_cols else None
    elif len(num_cols) >= 2:
        chart["type"] = "bar"
        chart["x"] = num_cols[0]
        chart["y"] = [num_cols[1]]
    else:
        chart["type"] = "table"

    append_msg(state, "assistant", f"ℹ️ Visualization plan: {json.dumps(chart, default=str)}")
    state["agent_results"]["visualization_plan"] = chart

    # Render accordingly
    try:
        if chart["type"] == "line":
            y_cols = chart["y"]
            x_col = chart["x"]
            title = " / ".join(y_cols) + " over " + x_col
            tool_plot_dataframe(pdf, x_col, y_cols, title=title)
            append_msg(state, "assistant", "✅ Line chart rendered.")
        elif chart["type"] == "bar":
            # simple bar: aggregate and plot
            x_col = chart["x"]
            y_col = chart["y"][0] if chart["y"] else None
            if y_col:
                agg = pdf.groupby(x_col)[y_col].sum().reset_index()
                display(agg.head(100))
                append_msg(state, "assistant", "✅ Bar chart data displayed as table (bar rendering optional).")
            else:
                display(pdf.head(100))
                append_msg(state, "assistant", "ℹ️ Displayed sample as table.")
        else:
            display(pdf.head(200))
            append_msg(state, "assistant", "ℹ️ Displayed table sample for analysis.")
        state["final_answer"] = "Visualization completed."
    except Exception as e:
        append_msg(state, "assistant", f"⚠️ Visualization failed: {e}")
        state["final_answer"] = f"Visualization failed: {e}"

    state["current_agent"] = "visualization_agent"
    state["next_agent"] = "supervisor"
    return state


In [0]:
# Cell 8 - Supervisor Agent (uses LLM optionally to decide next agent)
def supervisor_agent(state: dict) -> dict:
    curr = state.get("current_agent", "supervisor")
    append_msg(state, "assistant", f"🛰️ Supervisor invoked (current_agent={curr})")
    mapping = {
        "supervisor": {"next_agent": "data_analysis_agent", "reason": "start"},
        "data_analysis_agent": {"next_agent": "sql_agent", "reason": "analyzed"},
        "sql_agent": {"next_agent": "visualization_agent", "reason": "sql_ready"},
        "visualization_agent": {"next_agent": "END", "reason": "done"}
    }
    decision = mapping.get(curr, {"next_agent":"END", "reason":"default"})

    # Use safe serializable version of agent_results
    serial_agent_results = safe_serialize_agent_results(state.get("agent_results", {}))

    if USE_LLM_SUPERVISOR and llm is not None:
        prompt = (
            "You are a Databricks multi-agent supervisor. Available agents: "
            "data_analysis_agent, sql_agent, visualization_agent. "
            "Return strict JSON: {\"next_agent\":\"<agent_name>|END\",\"reason\":\"...\"}.\n\n"
            f"User query: {state['user_query']}\n\n"
            "Agent_results (safe summary):\n" + json.dumps(serial_agent_results, indent=2)[:2000]
        )
        try:
            resp = llm([HumanMessage(content=prompt)])
            txt = extract_text_from_llm_response(resp)
            s = txt.find("{"); e = txt.rfind("}") + 1
            if s != -1 and e != -1 and e > s:
                j = json.loads(txt[s:e])
                if "next_agent" in j:
                    decision = {"next_agent": j["next_agent"], "reason": j.get("reason", "chosen_by_llm")}
        except Exception as e:
            append_msg(state, "assistant", f"⚠️ LLM supervisor failed: {e}; falling back to deterministic.")

    append_msg(state, "assistant", f"Supervisor decided: {decision}")
    state["current_agent"] = "supervisor"
    state["next_agent"] = decision["next_agent"]
    return state


In [0]:
# Cell 9 - Build graph
workflow = StateGraph(MultiAgentState)

# Register supervisor node explicitly
workflow.add_node("supervisor", supervisor_agent)

# Add all agents from registry (names chosen in @register_agent)
for name, fn in _AGENT_REGISTRY.items():
    workflow.add_node(name, fn)

# Entry point
workflow.set_entry_point("supervisor")

# routing from supervisor based on next_agent
def route_after_supervisor(state: MultiAgentState) -> str:
    return state.get("next_agent", "END")

workflow.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "data_analysis_agent": "data_analysis_agent",
        "sql_agent": "sql_agent",
        "visualization_agent": "visualization_agent",
        "END": END
    }
)

# After each subagent return to supervisor
for name in _AGENT_REGISTRY.keys():
    workflow.add_edge(name, "supervisor")

multi_agent_system = workflow.compile()
print("✅ Databricks multi-agent system (3 agents) compiled.")


In [0]:
import json
import datetime

def default_json_serializer(obj):
    if isinstance(obj, (datetime.date, datetime.datetime)):
        return obj.isoformat()
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

def run_pipeline(user_query: str) -> MultiAgentState:
    state: MultiAgentState = {
        "messages": [{"role": "user", "content": user_query}],
        "user_query": user_query,
        "current_agent": "supervisor",
        "agent_results": {},
        "next_agent": "",
        "final_answer": ""
    }
    print("🚀 Running pipeline for query:", user_query)
    for k, v in state.items():
        if hasattr(v, "to_json"):
            state[k] = v.to_json()
        # If v is a date or datetime, convert to string
        elif isinstance(v, (datetime.date, datetime.datetime)):
            state[k] = v.isoformat()
    # If multi_agent_system.invoke internally serializes to JSON, pass the custom encoder
    final_state = multi_agent_system.invoke(state)
    print("\n--- Conversation (last messages) ---")
    for m in final_state.get("messages", []):
        print(f"{m.get('role')}: {str(m.get('content'))[:800]}\n{'-'*40}")
    print("\n--- Agent results keys ---", list(final_state.get("agent_results", {}).keys()))
    print("\n--- Final answer ---", final_state.get("final_answer"))
    return final_state

# Example run: update query and TARGET_TABLE as needed
res = run_pipeline("Show sales trends by product category over the last 2 years with a line chart")